# Week 6: Mini Project 模板

## 学习目标

1. 完成完整的 ML 项目流程
2. 应用前 5 周学到的所有技能
3. 培养问题解决能力
4. 学会撰写项目报告

## 项目背景

**问题**：预测单车共享系统的需求

**目标**：
1. 分析影响需求的因素
2. 构建需求预测模型
3. 提供运营建议

## 1. 数据加载与理解

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("项目工具已加载")

### 1.1 加载数据

In [ ]:
# 加载数据
# 如果没有真实数据，使用模拟数据
try:
    df = pd.read_csv('data/cleaned_bike_data.csv')
    print("成功加载真实数据")
except FileNotFoundError:
    # 生成模拟数据
    np.random.seed(42)
    n_samples = 1000
    
    stations = ['A001', 'A002', 'A003']
    station_id = np.random.choice(stations, n_samples)
    hour = np.random.randint(0, 24, n_samples)
    temperature = np.random.uniform(5, 35, n_samples)
    humidity = np.random.uniform(30, 90, n_samples)
    is_weekend = np.random.randint(0, 2, n_samples)
    is_holiday = np.random.randint(0, 2, n_samples)
    
    # 需求模型
    demand = (
        20 
        + 5 * (hour >= 7) & (hour <= 9)  # 早高峰
        + 5 * (hour >= 17) & (hour <= 19)  # 晚高峰
        + 0.5 * temperature 
        - 0.3 * humidity 
        - 10 * is_weekend
        + 15 * is_holiday
        + np.random.normal(0, 5, n_samples)
    )
    demand = np.maximum(demand, 0)  # 需求不能为负
    
    df = pd.DataFrame({
        'station_id': station_id,
        'hour': hour,
        'temperature': temperature,
        'humidity': humidity,
        'is_weekend': is_weekend,
        'is_holiday': is_holiday,
        'demand': demand
    })
    
    print("使用模拟数据")

print(f"数据形状: {df.shape}")
df.head()

### 1.2 数据概览

In [ ]:
# 基本统计
print("数据概览")
print("=" * 60)
print(f"样本数: {len(df)}")
print(f"特征数: {len(df.columns) - 1}")
print(f"\n缺失值:\n{df.isnull().sum()}")
print(f"\n数据类型:\n{df.dtypes}")

In [ ]:
# 描述性统计
df.describe()

## 2. 探索性数据分析（EDA）

### 2.1 单变量分析

In [ ]:
# 需求分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['demand'], bins=30, edgecolor='white', alpha=0.7)
axes[0].set_xlabel('需求量')
axes[0].set_ylabel('频数')
axes[0].set_title('需求分布')
axes[0].axvline(df['demand'].mean(), color='red', linestyle='--', label=f'均值={df["demand"].mean():.1f}')
axes[0].legend()

# 按小时的需求箱线图
df.boxplot(column='demand', by='hour', ax=axes[1])
axes[1].set_xlabel('小时')
axes[1].set_ylabel('需求量')
axes[1].set_title('各小时需求分布')
plt.suptitle('')

plt.tight_layout()
plt.show()

### 2.2 双变量分析

In [ ]:
# 相关性分析
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 相关性热力图
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, ax=axes[0])
axes[0].set_title('特征相关性')

# 温度 vs 需求
axes[1].scatter(df['temperature'], df['demand'], alpha=0.5)
axes[1].set_xlabel('温度 (°C)')
axes[1].set_ylabel('需求量')
axes[1].set_title('温度 vs 需求')

plt.tight_layout()
plt.show()

### 2.3 多变量分析

In [ ]:
# 按站点和时间段分析
pivot_data = df.pivot_table(values='demand', index='hour', columns='station_id', aggfunc='mean')

fig, ax = plt.subplots(figsize=(12, 6))

for col in pivot_data.columns:
    ax.plot(pivot_data.index, pivot_data[col], marker='o', label=col)

ax.set_xlabel('小时')
ax.set_ylabel('平均需求')
ax.set_title('各站点 24 小时需求曲线')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. 特征工程

In [ ]:
# 创建新特征
df['is_rush_hour'] = ((df['hour'] >= 7) & (df['hour'] <= 9) | 
                      (df['hour'] >= 17) & (df['hour'] <= 19)).astype(int)

df['temp_squared'] = df['temperature'] ** 2

# 时间段分类
def categorize_hour(h):
    if 7 <= h <= 9:
        return 'morning_rush'
    elif 17 <= h <= 19:
        return 'evening_rush'
    elif 10 <= h <= 16:
        return 'daytime'
    else:
        return 'night'

df['time_period'] = df['hour'].apply(categorize_hour)

print("新特征创建完成")
df.head()

## 4. 模型训练

### 4.1 数据准备

In [ ]:
# 特征和目标
feature_cols = ['hour', 'temperature', 'humidity', 'is_weekend', 'is_holiday', 'is_rush_hour']

X = df[feature_cols]
y = df['demand']

# 分割数据
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"训练集: {X_train.shape}")
print(f"测试集: {X_test.shape}")

### 4.2 基线模型

In [ ]:
# 基线：线性回归
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("基线模型：线性回归")
print("=" * 40)
print(f"R²: {r2_score(y_test, y_pred_lr):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred_lr):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lr)):.2f}")

### 4.3 模型比较

In [ ]:
# 比较多个模型
models = {
    '线性回归': LinearRegression(),
    'Ridge': Ridge(alpha=1),
    '随机森林': RandomForestRegressor(n_estimators=100, random_state=42),
    '梯度提升': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    
    results.append({
        '模型': name,
        '测试 R²': r2_score(y_test, y_pred),
        '测试 MAE': mean_absolute_error(y_test, y_pred),
        'CV R²': np.mean(cv_scores)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

### 4.4 超参数调优

In [ ]:
# 随机森林调优
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10]
}

rf = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("最佳参数:", grid_search.best_params_)
print("最佳 CV R²:", f"{grid_search.best_score_:.4f}")

In [ ]:
# 最终模型
best_model = grid_search.best_estimator_
y_pred_final = best_model.predict(X_test)

print("最终模型性能")
print("=" * 40)
print(f"R²: {r2_score(y_test, y_pred_final):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred_final):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_final)):.2f}")

## 5. 结果分析与解释

In [ ]:
# 特征重要性
importance = pd.DataFrame({
    '特征': feature_cols,
    '重要性': best_model.feature_importances_
}).sort_values('重要性', ascending=False)

print("特征重要性排名")
print(importance.to_string(index=False))

In [ ]:
# 可视化特征重要性
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 特征重要性条形图
axes[0].barh(importance['特征'], importance['重要性'])
axes[0].set_xlabel('重要性')
axes[0].set_title('特征重要性')

# 预测 vs 实际
axes[1].scatter(y_test, y_pred_final, alpha=0.5)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('实际需求')
axes[1].set_ylabel('预测需求')
axes[1].set_title('预测 vs 实际')

plt.tight_layout()
plt.show()

## 6. 结论与建议

### 6.1 主要发现

1. **时间因素**：高峰时段需求显著高于其他时段
2. **温度影响**：温度与需求呈正相关
3. **周末效应**：周末需求低于工作日
4. **站点差异**：不同站点需求模式不同

### 6.2 运营建议

1. **车辆调度**：在高峰时段前增加热门站点车辆
2. **维护时间**：选择低需求时段（夜间）进行维护
3. **站点规划**：根据需求模式优化站点位置
4. **促销策略**：在低需求时段推出优惠活动

### 6.3 模型局限

1. 未考虑天气变化（雨雪等）
2. 未考虑突发事件（大型活动等）
3. 数据时间跨度有限
4. 未考虑用户行为变化

## 7. 后续改进

### 7.1 可以尝试的方向

1. 加入更多特征（天气、POI 等）
2. 尝试深度学习模型
3. 时间序列建模
4. 异常值检测
5. 实时预测系统

## 8. 总结

### 项目完成清单

- [x] 数据加载与理解
- [x] 探索性数据分析
- [x] 特征工程
- [x] 模型训练与调优
- [x] 结果分析
- [x] 结论与建议

### 学习要点

1. **完整的 ML 流程**：从数据到部署
2. **EDA 的重要性**：理解数据是建模的基础
3. **特征工程**：好的特征比复杂的模型更重要
4. **模型解释**：不仅要预测，还要理解原因